In [ ]:
import sqlite3

conn = sqlite3.connect("../data/travel.sqlite")

cursor = conn.cursor()

In [2]:

# 1. List all tables
print("TABLES:")
cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type='table';
""")
for row in cursor.fetchall():
    print(row[0])

print("\n---\n")

# 2. Inspect flights table structure (like df.info())
print("FLIGHTS TABLE STRUCTURE:")
cursor.execute("PRAGMA table_info(flights);")
for col in cursor.fetchall():
    print(col)

print("\n---\n")

# 3. Peek at first 5 rows (like df.head())
print("FLIGHTS SAMPLE ROWS:")
cursor.execute("SELECT * FROM flights LIMIT 5;")
for row in cursor.fetchall():
    print(row)



TABLES:
aircrafts_data
airports_data
boarding_passes
bookings
flights
seats
ticket_flights
tickets

---

FLIGHTS TABLE STRUCTURE:
(0, 'flight_id', 'INTEGER', 1, None, 0)
(1, 'flight_no', 'character(6)', 1, None, 0)
(2, 'scheduled_departure', 'timestamp with time zone', 1, None, 0)
(3, 'scheduled_arrival', 'timestamp with time zone', 1, None, 0)
(4, 'departure_airport', 'character(3)', 1, None, 0)
(5, 'arrival_airport', 'character(3)', 1, None, 0)
(6, 'status', 'character varying(20)', 1, None, 0)
(7, 'aircraft_code', 'character(3)', 1, None, 0)
(8, 'actual_departure', 'timestamp with time zone', 0, None, 0)
(9, 'actual_arrival', 'timestamp with time zone', 0, None, 0)

---

FLIGHTS SAMPLE ROWS:
(1185, 'PG0134', '2017-09-10 09:50:00+03', '2017-09-10 14:55:00+03', 'DME', 'BTK', 'Scheduled', '319', '\\N', '\\N')
(3979, 'PG0052', '2017-08-25 14:50:00+03', '2017-08-25 17:35:00+03', 'VKO', 'HMA', 'Scheduled', 'CR2', '\\N', '\\N')
(4739, 'PG0561', '2017-09-05 12:30:00+03', '2017-09-05 14:15:0

In [3]:
# STEP 2: basic dataset sanity

# total number of flights
cursor.execute("SELECT COUNT(*) FROM flights;")
print("Total flights:", cursor.fetchone()[0])

print("\nFlights by status:")

cursor.execute("""
SELECT
    status,
    COUNT(*) AS flights
FROM flights
GROUP BY status
ORDER BY flights DESC;
""")

for row in cursor.fetchall():
    print(row)
print("\n---\n")


Total flights: 33121

Flights by status:
('Arrived', 16707)
('Scheduled', 15383)
('On Time', 518)
('Cancelled', 414)
('Departed', 58)
('Delayed', 41)

---



In [4]:
# STEP 3: check usable actual timestamps

cursor.execute("""
SELECT
    COUNT(*) AS total_flights,
    SUM(CASE WHEN actual_departure != '\\N' THEN 1 ELSE 0 END) AS has_actual_departure,
    SUM(CASE WHEN actual_arrival  != '\\N' THEN 1 ELSE 0 END) AS has_actual_arrival
FROM flights;
""")

print(cursor.fetchone())


(33121, 16773, 16715)


In [5]:
# STEP 4: average departure delay in minutes

cursor.execute("""
SELECT
    ROUND(
        AVG(
            (JULIANDAY(SUBSTR(actual_departure, 1, 19))
           - JULIANDAY(SUBSTR(scheduled_departure, 1, 19))) * 1440
        ),
        2
    ) AS avg_dep_delay_min
FROM flights
WHERE status != 'Cancelled'
  AND actual_departure != '\\N';
""")

print("Average departure delay (minutes):", cursor.fetchone()[0])


Average departure delay (minutes): 12.2


In [6]:
# STEP 5: average arrival delay (minutes)

cursor.execute("""
SELECT
    ROUND(
        AVG(
            (JULIANDAY(SUBSTR(actual_arrival, 1, 19))
           - JULIANDAY(SUBSTR(scheduled_arrival, 1, 19))) * 1440
        ),
        2
    ) AS avg_arr_delay_min
FROM flights
WHERE status != 'Cancelled'
  AND actual_arrival != '\\N';
""")

print("Average arrival delay (minutes):", cursor.fetchone()[0])


Average arrival delay (minutes): 12.2


In [7]:
conn.close()

## 📌 Data Understanding & Validation Summary

### What we did

In this notebook, we focused on understanding and validating the airline operations dataset before doing any analytics or dashboard work.

The dataset was provided as a SQLite database (`travel.sqlite`), not as CSV files. Instead of loading data into pandas, we connected directly to the database using Python and the `sqlite3` library, and explored the data using pure SQL queries.

The goal of this phase was to ensure that:
- The dataset structure is clear
- The data is usable for delay analysis
- Time-based calculations work correctly
- No incorrect assumptions are made before building SQL views

---

### Dataset structure

- The database contains multiple tables related to airline operations.
- The `flights` table is the core fact table.
- Each row in the `flights` table represents one flight.
- Key columns include:
  - Scheduled and actual departure and arrival timestamps
  - Departure and arrival airports
  - Flight status
  - Aircraft code

The table structure was confirmed using `PRAGMA table_info` instead of assuming column names.

---

### Key data quality findings

- Missing actual timestamps are stored as the string `'\N'`, not as SQL NULL values.
- A large portion of flights have status `Scheduled`, meaning they did not actually operate.
- Only about half of the flights have usable actual departure and arrival timestamps.
- Cancelled flights exist and must be excluded from delay calculations.

These findings confirm that the dataset is realistic and reflects real airline operational data.

---

### Time and delay handling

- Timestamps include timezone information (for example, `+03`).
- SQLite’s date and time functions cannot directly parse timezone strings.
- To compute delays correctly, the timezone part of the timestamp was removed using `SUBSTR`.
- Delays were calculated as the difference between actual and scheduled times, expressed in minutes.
- Delay calculations were performed only for flights that actually operated.

---

### Initial KPI results (sanity check)

As a validation step, basic delay metrics were computed:

- **Average departure delay:** approximately 12.2 minutes  
- **Average arrival delay:** approximately 12.2 minutes  

This suggests that:
- Delays are not significantly recovered during flight
- Delay propagation exists across the operation
- The data behaves in a realistic and expected manner

---

### Why this step matters

This data understanding and validation phase ensures that:
- All future KPIs are built on correct and verified logic
- Delay calculations are reliable and reproducible
- SQL remains the single source of truth
- The project follows real-world analytics practices

With these checks complete, the next step is to build reusable SQL views that will be used for Excel validation and Tableau dashboards.
